# Project Group: 10 

Members: Mulham Omar, Jalissa Rattan, Benjamin Rider, Hinke Verbaan, Lois Zhao

# Research Objective

Compare how demographic data affects bus ridership in Boston

Sub-questions:
- How does median age of the local block group affect bus stop ridership?
- How does the gender distribution of the local block group affect bus stop ridership?
- How does population density of the local block group affect bus stop ridership?

# Contribution Statement


**Mulham**: Streamlit visualization

**Jalissa**: Make charts and collect/aggregate results

**Ben**: Collect dataset/background research, intial cleaning/analysis pipeline

**Hinke**: Create final code that runs all required functions

**Lois**: Statistical analysis

**All** Write their sections of the final report

# Data Used

Dataset 1: MBTA Bus Ridership by Time Period, Season, Route/Line, and Stop - Fall (2024)
https://mbta-massdot.opendata.arcgis.com/datasets/7acd353c1a734eb8a23caf46a0e66b23_0/explore?filters=eyJzZWFzb24iOlsiRmFsbCAyMDI0IiwiRmFsbCAyMDIwIl19

Dataset 2: MassGIS Data: 2020 U.S. Census
https://www.mass.gov/info-details/massgis-data-2020-us-census

Dataset 3: Jonathan Schroeder, David Van Riper, Steven Manson, Katherine Knowles, Tracy Kugler, Finn Roberts, and Steven Ruggles. IPUMS National Historical Geographic Information System: Version 20.0 [dataset]. Minneapolis, MN: IPUMS. 2025. http://doi.org/10.18128/D050.V20.0

Dataset 4: MassGIS Data: MBTA Bus Routes and Stops
https://www.mass.gov/info-details/massgis-data-mbta-bus-routes-and-stops

# Data Pipeline

① Input data: We extract the datasets from the websites (mentioned in Data Used). 

Step 1: Demographic data from Dataset 3 is paired with its Census Block Group shapefile from Dataset 2. 
Step 2: A spatial join is performed in ArcGIS Pro to pair each bus stop with its Census Block Group
Step 3: Each entry in Dataset 1 is paired with demographic data for its respective bus stop.
Step 4: The table is exported from ArcGIS Pro into an XLSX file, then exported to a CSV using Excel.

② Transform data: In this step we select and transform useful data from the datasets. Below here, we describe how we get the dataframe for the analysis analyse. 


- Bus_Ridership: **Ben can you add how you did this in gis?** In dataset 1, the usage is devided by time of day, for our research question we need to calculate the average use of a busstop in general **per day**. Therefore from database 1 we add average_ons and average_off per stop_ID this is devided by the amount of times that this specific stop_ID was in the database. This transformation will be done in python.

                       Busridership per day for specific busstop == total_trips/amount of entries in stop_ID

Where total_trips is:

                                     total_flow == [average_ons + average_offs] * total_trips


All of the datatypes below were transformed via the computerprogramm arcgis. With intersecting the busstop location(dataset 1) and the local block group (dataset 2). From here the data was translated into a csv file. 

- median_age: The median age in a certain local block group. This datatype was directly selected from dataset 2. We do not need to do further processing in Python.

- gender_distribution(Percent Male/Percent Female): The gender distribution in a certain local block group. This datatype was directly selected from dataset 2. We do not need to do further processing in Python.

- population_density: The total population (Total_Popu) of a single block group we can get this directly from dataset 2. The area of a single block group (Block_Area **what is the metric**) can be derived from dataset 3. This transformation will be done in python.

                                         Population density == Total_Popu / Block_Area


③ Analyze data: For every research question the same statistical analysis is used, as we are just switching what we are comparing the ridership data to (Median_Age, Gender_Distribution and Population_Density). This will be done by using the linear regrssion, Spearman correlation and Anova analysis respectively for each  sub-question in python.

**Transformation**

In [1]:
#Relevant packages
import pandas as pd
import numpy as np

In [15]:
# #What are the columns in the file (not nessesary for the code but for a more clear view)
# def get_columns(input_df):
#     """ Returns the available column names from the input DataFrame in a set. """
#     # Use only 'input_df' inside this function, not 'df'
#     # YOUR CODE HERE
#     return set(input_df.columns)

# columns = get_columns(df)
# print(type(columns)) # should print "<class 'set'>"
# print('Columns: ' + str(columns))

In [4]:
df = pd.read_csv("TIL6022Data.csv") #reading the file
def calculate_columns(df):
    df['Pop_Density'] = df['Total_Pop']/df['BG Area']
    df['Total_Trips'] = (df['num_trips']*5).where(df['day_type_name'] == 'weekday', df['num_trips'])

#agg_functions = {'total_trips': 'mean', 'median_age': 'mean', 'popdensity' : 'mean', 'percent_male': 'mean'} #transformations done in the aggregate funciton

#df_new = df.groupby(df['stop_id']).aggregate(agg_functions) #transforming the data
#print(df_new)]
calculate_columns(df)
print(df)

       route_id  direction_id day_type_name time_period_name  \
0             1             0       weekday      MIDDAY_BASE   
1             1             0       weekday         EARLY_AM   
2             1             0       weekday          PM_PEAK   
3             1             0       weekday          PM_PEAK   
4             1             0       weekday      MIDDAY_BASE   
...         ...           ...           ...              ...   
105913       99             1        sunday         OFF_PEAK   
105914       99             1        sunday         OFF_PEAK   
105915       99             1        sunday         OFF_PEAK   
105916       99             1        sunday         OFF_PEAK   
105917       99             1        sunday         OFF_PEAK   

                            stop_name  stop_id  average_ons  average_offs  \
0       MASSACHUSETTS AVE @ MASSACHUS      188          3.9           6.2   
1       MASSACHUSETTS AVE @ WASHINGTO    10590          1.7           0.6   


Spearman statistical analysis

In [43]:
from scipy.stats import spearmanr



def spearmann_result(a,b):
    rho, p = spearmanr(df_new[a], df_new[b])
    if p >= 0.05:
        print(f"There is no statistically significant correlation between {a} and {b}")
    else:
        print(f"The correlation between {a} and {b} is statistically significant.")
        if rho <= 0.20 and rho >=-0.20: 
            print(f"But with a rho of {round(rho,3)} the correlation between {a} and {b} is negligible.")
        elif rho <= 0.40 and rho >=-0.40: 
            print(f"But with a rho of {round(rho,3)} the correlation between {a} and {b} is weak.")
        elif rho <= 0.60 and rho >=-0.60: 
            print(f"With a rho of {round(rho,3)} the correlation between {a} and {b} is moderate.")
        elif rho <= 0.80 and rho >=-0.80: 
            print(f"With a rho of {round(rho,3)} the correlation between {a} and {b} is strong.")
        else: 
            print(f"With a rho of {round(rho,3)} the correlation between {a} and {b} is very strong.")
    return



spearmann_result('percent_male','total_trips')

The correlation between percent_male and total_trips is statistically significant.
But with a rho of -0.058 the correlation between percent_male and total_trips is negligible.
